In [1]:
#Import Required Libraries & Setup
import torch
import torch.nn as nn
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
import warnings
warnings.filterwarnings("ignore")
# Automatically select the best available hardware: GPU (CUDA) if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# Initialize the ResNet50 Model

In [2]:
# Load ResNet50 with pre-trained weights from ImageNet
cnn_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
# Freeze the early layers (layer1, layer2, layer3) to retain general features learned from ImageNet
# This prevents overfitting and speeds up training
for name, param in cnn_model.named_parameters():
    if 'layer1' in name or 'layer2' in name or 'layer3' in name:
        param.requires_grad = False
# Get the number of input features for the final fully connected layer
num_ftrs = cnn_model.fc.in_features
# Replace the original ImageNet classifier with a custom one for our binary classification task
# Dropout(0.5) is added to prevent overfitting by randomly turning off half the neurons
cnn_model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(num_ftrs, 2))
# Move the entire model to the selected device (GPU/CPU)
cnn_model = cnn_model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s]


#  Define Data Augmentation Pipelines

In [3]:
train_transform = A.Compose([
    A.Resize(224, 224),# Resize all images to the input size expected by ResNet50
    A.HorizontalFlip(p=0.5), # Flip images horizontally with 50% probability
    A.RandomRotate90(p=0.2),# Rotate images 90 degrees randomly with 20% probability
    A.ShiftScaleRotate(0.1, 0.1, 15, p=0.5),# Randomly shift, scale, and rotate images
    A.RandomBrightnessContrast(p=0.3),# Randomly adjust brightness and contrast
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),# Normalize using ImageNet stats
    ToTensorV2()  # Convert the image to a PyTorch tensor
])
# Validation/Test transforms: Only apply deterministic operations (NO random augmentations!)
val_test_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Unzip the Competition Data

In [4]:
import zipfile
import os


train_zip_path = '/kaggle/input/competitions/dogs-vs-cats-redux-kernels-edition/train.zip'
train_extract_path = '/kaggle/working/train/'

os.makedirs(train_extract_path, exist_ok=True)
with zipfile.ZipFile(train_zip_path, 'r') as zip_ref:
    zip_ref.extractall(train_extract_path)


test_zip_path = '/kaggle/input/competitions/dogs-vs-cats-redux-kernels-edition/test.zip'
test_extract_path = '/kaggle/working/test/'

os.makedirs(test_extract_path, exist_ok=True)
with zipfile.ZipFile(test_zip_path, 'r') as zip_ref:
    zip_ref.extractall(test_extract_path)

#  Create a Custom Dataset Class

In [5]:
class DogsCatsDataset(Dataset):
    def __init__(self, data_dir, transform, data_type="train"):
        self.data_type = data_type
        self.transform = transform
        if data_type == "train":
            img_dir = os.path.join(data_dir, "train", "train")
        else:
            # Create a list of all valid image file paths in the directory
            img_dir = os.path.join(data_dir, "test", "test")
        self.image_paths = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith(".jpg")]
        # For training data: Create labels (0 for cat, 1 for dog) based on the filename
        if data_type == "train":
            self.labels = [0 if "cat" in f else 1 for f in self.image_paths]
            # For test data: Extract the image ID from the filename (needed for the submission file)
        else:
            self.test_ids = [int(os.path.splitext(os.path.basename(f))[0]) for f in self.image_paths]

    def __len__(self):                              #1. Returns the total number of images in the dataset
        return len(self.image_paths)

    def __getitem__(self, idx):                    # 2. Apply the data augmentation transforms
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        if self.transform:
            image = self.transform(image=image)["image"]
        if self.data_type == "train":
            return image, self.labels[idx]
        else:
            return image, self.test_ids[idx]


# Prepare DataLoaders for Training and Validation

In [6]:
data_root = "/kaggle/working/"
train_dataset = DogsCatsDataset(data_root, train_transform, "train")
val_dataset = DogsCatsDataset(data_root, val_test_transform, "train")

indices = list(range(len(train_dataset)))
train_idx=list(range(20000))
val_idx=list(range(20000,25000))

train_dataset = torch.utils.data.Subset(train_dataset, train_idx)
val_dataset = torch.utils.data.Subset(val_dataset, val_idx)

train_dl = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=8, pin_memory=True,prefetch_factor=2,persistent_workers=True)
val_dl = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True,persistent_workers=True)


# Define Loss Function, Optimizer, and Scheduler

In [7]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.amp import GradScaler
loss_func = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW([
    {'params': cnn_model.layer4.parameters(), 'lr': 1e-4},
    {'params': cnn_model.fc.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)
lr_scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
scaler = GradScaler()

# Define Training and Validation Logic

In [8]:
def loss_batch(loss_func, output, target, opt=None, scaler=None):
    if opt is not None:
        with autocast():
            loss = loss_func(output, target)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    else:
        loss = loss_func(output, target)
    pred = output.argmax(dim=1, keepdim=True)
    metric_b = pred.eq(target.view_as(pred)).sum().item()
    return loss.item(), metric_b

def loss_epoch(model, loss_func, dataset_dl, opt=None, scaler=None):
    run_loss, t_metric = 0.0, 0.0
    len_data = len(dataset_dl.dataset)
    for xb, yb in dataset_dl:
        xb, yb = xb.to(device), yb.to(device)
        output = model(xb)
        loss_b, metric_b = loss_batch(loss_func, output, yb, opt, scaler)
        run_loss += loss_b
        t_metric += metric_b
    return run_loss / len_data, t_metric / len_data

def train_val(model, epochs=30, patience=5):
    best_loss = float("inf")
    early_stop_count = 0
    for epoch in range(epochs):
        model.train()
        train_loss, train_acc = loss_epoch(model, loss_func, train_dl, optimizer, scaler)
        model.eval()
        with torch.no_grad():
            val_loss, val_acc = loss_epoch(model, loss_func, val_dl)
        lr_scheduler.step()
        print(f"Epoch {epoch}: Train Loss {train_loss:.4f}, Acc {train_acc:.4f} | Val Loss {val_loss:.4f}, Acc {val_acc:.4f}")
        if val_loss < best_loss:
            best_loss = val_loss
            early_stop_count = 0
            torch.save(model.state_dict(), "best_resnet.pt")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                print("early stopping has started")
                break
    model.load_state_dict(torch.load("best_resnet.pt"))
    return model

# Start the training process
cnn_model = train_val(cnn_model)

Epoch 0: Train Loss 0.0038, Acc 0.9780 | Val Loss 0.0034, Acc 0.9944
Epoch 1: Train Loss 0.0034, Acc 0.9922 | Val Loss 0.0033, Acc 0.9956
Epoch 2: Train Loss 0.0034, Acc 0.9938 | Val Loss 0.0033, Acc 0.9958
Epoch 3: Train Loss 0.0033, Acc 0.9961 | Val Loss 0.0033, Acc 0.9964
Epoch 4: Train Loss 0.0033, Acc 0.9963 | Val Loss 0.0033, Acc 0.9968
Epoch 5: Train Loss 0.0033, Acc 0.9951 | Val Loss 0.0033, Acc 0.9952
Epoch 6: Train Loss 0.0033, Acc 0.9953 | Val Loss 0.0033, Acc 0.9948
Epoch 7: Train Loss 0.0033, Acc 0.9970 | Val Loss 0.0033, Acc 0.9926
Epoch 8: Train Loss 0.0032, Acc 0.9971 | Val Loss 0.0033, Acc 0.9950
Epoch 9: Train Loss 0.0032, Acc 0.9981 | Val Loss 0.0033, Acc 0.9956
early stopping has started


# Test Time Augmentation (TTA) for Final Prediction

In [9]:
test_dataset = DogsCatsDataset(data_root, val_test_transform, "test")
test_dl = DataLoader(test_dataset, batch_size=64, shuffle=False)

def predict_with_tta(model, test_dl, n_tta=5):
    model.eval()
    all_ids, all_probs = [], []
    with torch.no_grad():
        for imgs, ids in test_dl:
            imgs = imgs.to(device)
            batch_probs = []
            for _ in range(n_tta):
                outputs = model(imgs)
                probs = torch.softmax(outputs, dim=1)[:,1].cpu().numpy()
                batch_probs.append(probs)
            avg_probs = np.mean(batch_probs, axis=0)
            all_ids.extend(ids.numpy())
            all_probs.extend(avg_probs)
    return all_ids, all_probs

test_ids, test_probs = predict_with_tta(cnn_model, test_dl)
pd.DataFrame({"id": test_ids, "label": test_probs}).to_csv("submission_tta.csv", index=False)